<a href="https://colab.research.google.com/github/profbnk/AI/blob/main/laya_study_plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install laya

In [2]:
from laya import Router

router = Router()  # downloads a checkpoint on first use; Router(preload=True) loads all three up front

state = "Hi, we were billed twice for March. Please refund the duplicate today or we will cancel our plan."
questions = {
    "department": {"type": "choice", "instructions": "Which department should handle this?",
                   "criteria": {"billing": "invoices, payments, refunds",
                                "technical": "bugs, outages, system errors",
                                "other": "everything else"}},
    "urgency": {"type": "score", "instructions": "How urgent is this?",
                "criteria": ["not urgent", "soon", "blocking"]},
    "churn_risk": {"type": "noul", "instructions": "Does the user threaten to cancel or leave?"},
}

result = router.predict(state, questions)
print(result["answers"]["department"]["choice"])  # billing
print(result["answers"]["churn_risk"]["noul"])    # probability the answer is yes
print(result["routing"]["model"])                 # english

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/laya/router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


billing
0.879
english


In [ ]:
for text in ["मुझसे मार्च में दो बार शुल्क लिया गया, कृपया डुप्लिकेट राशि वापस करें।",
             "La aplicación se cierra cada vez que abro la configuración."]:
    r = router.predict(text, {"department": questions["department"]})
    print(r["routing"]["model"], r["answers"]["department"]["choice"])
# multilingual billing
# multilingual technical


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

multilingual billing
multilingual technical


In [3]:
# Paste this whole thing into ONE Colab cell, below your Laya code.
import ipywidgets as widgets
from IPython.display import display, clear_output


# ---------------------------------------------------------------
# 1. PLUG IN YOUR LAYA CODE HERE
# ---------------------------------------------------------------
from laya import Router

router = Router()  # loads once; reused on every button click


def laya_decide(state, question):
    # Laya reads the state as plain text
    state_text = (
        f"Student's weakest subject is {state['weak_subject']}. "
        f"Exam in {state['days_left']} days. "
        f"Free time today: {state['hours_free']} hours. "
        f"Confidence in the weak subject: {state['weak_confidence']} out of 10. "
        f"Current energy: {state['energy']} out of 10."
    )
    questions = {
        "action": {
            "type": "choice",
            "instructions": question,
            "criteria": {
                "Revise weak subject": "exam is not very close and confidence is low, so focus on the weakest subject",
                "Practice past papers": "exam is a few weeks away and confidence is decent, so practice questions",
                "Take a mock test": "exam is within about a week, so test under exam conditions",
                "Light revision": "exam is within a day or two, so only quick and relaxed revision",
                "Take a break": "energy is very low or almost no free time, so rest",
            },
        }
    }
    result = router.predict(state_text, questions)
    return result["answers"]["action"]["choice"]


ACTIONS = {
    "Revise weak subject": "Deep-focus on your weakest subject (45 min blocks).",
    "Practice past papers": "Solve previous-year questions under a time limit.",
    "Take a mock test": "Attempt a full mock test and review mistakes.",
    "Light revision": "Quick notes and formulas revision. Keep it relaxed.",
    "Take a break": "Rest, walk, sleep. A fresh brain learns faster.",
}


def rule_based_decide(s):
    if s["energy"] <= 3 or s["hours_free"] < 1:
        return "Take a break"
    if s["days_left"] <= 2:
        return "Light revision"
    if s["days_left"] <= 7:
        return "Take a mock test"
    if s["weak_confidence"] <= 4:
        return "Revise weak subject"
    return "Practice past papers"


# ---------------------------------------------------------------
# 2. SIMPLE UI (sliders + button, all inside the notebook)
# ---------------------------------------------------------------
subject = widgets.Text(value="Data Structures", description="Weak subject")
days = widgets.IntSlider(value=10, min=0, max=60, description="Days left")
hours = widgets.FloatSlider(value=3, min=0, max=12, step=0.5, description="Free hours")
conf = widgets.IntSlider(value=4, min=1, max=10, description="Confidence")
energy = widgets.IntSlider(value=6, min=1, max=10, description="Energy")
button = widgets.Button(description="Ask Laya", button_style="success")
out = widgets.Output()
history = []


def on_click(_):
    state = {
        "weak_subject": subject.value,
        "days_left": days.value,
        "hours_free": hours.value,
        "weak_confidence": conf.value,
        "energy": energy.value,
    }
    decision = laya_decide(state, "What should the student do next?")
    baseline = rule_based_decide(state)
    history.append((days.value, hours.value, energy.value, decision, baseline))

    with out:
        clear_output()
        print("LAYA DECISION :", decision)
        print("ADVICE        :", ACTIONS.get(decision, ""))
        print("RULE-BASED    :", baseline,
              "(agrees)" if baseline == decision else "(different! why?)")
        print("\nPast decisions (days, hours, energy -> Laya | rules):")
        for d, h, e, l, b in history[-5:]:
            print(f"  {d}d, {h}h, energy {e} -> {l} | {b}")


button.on_click(on_click)
display(widgets.VBox([subject, days, hours, conf, energy, button, out]))


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]